# Patient Recommender System

Using RAG framework to retrieve cases and provide summaries for patients that are of similar nature

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 30/03/2026   | Martin | Create  | Notebook created. Started working on document format for patient profile | 

# Content

* [Patient Snapshot](#patient-snapshot)

# Patient Snapshot

Explore the availability of data from both Synthea and MIMIC resources to craft the patient's profile

In [1]:
import duckdb

In [2]:
con = duckdb.connect("../data/warehouse/mimic_fhir.duckdb")

In [4]:
con.sql("SHOW TABLES FROM silver;")

┌───────────────────────────────────┐
│               name                │
│              varchar              │
├───────────────────────────────────┤
│ condition                         │
│ encounter                         │
│ location                          │
│ medication_dispense               │
│ obs_vitals                        │
│ organization                      │
│ patient                           │
│ procedure                         │
│ synthea_allergy_intolerance       │
│ synthea_careplan                  │
│ synthea_careteam                  │
│ synthea_condition                 │
│ synthea_encounter                 │
│ synthea_immunization              │
│ synthea_medication                │
│ synthea_medication_administration │
│ synthea_medication_request        │
│ synthea_observation               │
│ synthea_patient                   │
│ synthea_procedure                 │
├───────────────────────────────────┤
│              20 rows              │
└───────────

## MIMIC Data

Observations from patients:

1. f040bdf0-6951-5194-a400-fac7a247583c (15174162)
2. f7ed2ddf-9129-51d0-9bad-0d53665559db (15653234)
3. d4566ed7-5ee5-5cb1-bb88-b08f5b0a8c03 (15653252)

<u>Data Present</u>

| Column | Table | Type |  
| ------ | ----- | ---- |
| name | patient | categorical |
| gender | patient | categorical |
| birth_date | patient | date (YYYY-DD-MM) |
| race | patient | categorical |
| ethnicity | patient | categorical |
| marital_status | patient | categorical |

In [7]:
id1 = "f040bdf0-6951-5194-a400-fac7a247583c"
id2 = "f7ed2ddf-9129-51d0-9bad-0d53665559db"
id3 = "d4566ed7-5ee5-5cb1-bb88-b08f5b0a8c03"

In [28]:
con.sql(f"SELECT * FROM silver.condition WHERE patient_id='{id1}';")

┌──────────────────────────────────────┬──────────┬─────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┐
│             resource_id              │ icd_code │                                    icd_name                                     │              patient_id              │ condition_category  │             encounter_id             │
│               varchar                │ varchar  │                                     varchar                                     │               varchar                │       varchar       │               varchar                │
├──────────────────────────────────────┼──────────┼─────────────────────────────────────────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼──────────────────────────────────────┤
│ ff4a1fc3-16ff-5f20-ac65-2f08531c7fd0 │ R0902    │ Hypoxemia   

In [22]:
con.sql("SELECT DISTINCT condition_category FROM silver.condition;")

┌─────────────────────┐
│ condition_category  │
│       varchar       │
├─────────────────────┤
│ encounter-diagnosis │
└─────────────────────┘

In [27]:
con.sql(f"SELECT * FROM silver.encounter WHERE patient_id='{id1}' LIMIT 10;")

┌──────────────────────────────────────┬────────────┬────────────┬─────────────────────┬─────────────────────┬──────────┬──────────────────────────────────────┬──────────────┬───────────────────────┬──────────────────────────────────────┐
│             resource_id              │ class_code │ class_name │   start_timestamp   │    end_timestamp    │  status  │              patient_id              │ admit_source │ discharge_disposition │           organization_id            │
│               varchar                │  varchar   │  varchar   │      timestamp      │      timestamp      │ varchar  │               varchar                │   varchar    │        varchar        │               varchar                │
├──────────────────────────────────────┼────────────┼────────────┼─────────────────────┼─────────────────────┼──────────┼──────────────────────────────────────┼──────────────┼───────────────────────┼──────────────────────────────────────┤
│ 25f36802-6030-5cbc-bbf1-b629ebd7bef9 │ EME

In [29]:
con.sql(f"SELECT * FROM silver.procedure WHERE patient_id='{id1}' LIMIT 10;")

┌──────────────────────────────────────┬──────────────┬───────────────────────────────────────┬───────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┐
│             resource_id              │ snomed_ct_id │          snomed_ct_procedure          │  status   │              patient_id              │             encounter_id             │ performed_datetime  │
│               varchar                │   varchar    │                varchar                │  varchar  │               varchar                │               varchar                │      timestamp      │
├──────────────────────────────────────┼──────────────┼───────────────────────────────────────┼───────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┤
│ a78e8c88-913e-5ff7-9ae2-e6aa50e14e7f │ 410188000    │ Taking patient vital signs assessment │ completed │ f040bdf0-6951-5194-a400-fac7a247583c │ 25f36802-6030-5cbc-bb

## Synthea Data

Observations from patients:

1. 24283ca3-2f37-a4c1-e740-97f060a230a6
2. 2e312e34-cc56-6a67-6642-c261c2f4838f
3. a3d61946-69d4-cd45-0792-f8b941bd6f73

<u>Data Present</u>

| Column | Table | Type | Notes | 
| ------ | ----- | ---- | ----- |
| name | patient | categorical | Combine first_name + middle_name + last_name |
| gender | patient | categorical | |
| birth_date | patient | date (YYYY-DD-MM) | |
| race | patient | categorical | |
| ethnicity | patient | categorical | |
| marital_status | patient | categorical | Convert to MIMIC format |


In [33]:
syn_id1 = "24283ca3-2f37-a4c1-e740-97f060a230a6"
syn_id2 = "2e312e34-cc56-6a67-6642-c261c2f4838f"
syn_id3 = "a3d61946-69d4-cd45-0792-f8b941bd6f73"

In [13]:
con.sql("SELECT * FROM silver.synthea_patient;")

┌──────────────────────────────────────┬─────────────┬─────────────┬────────────┬─────────┬───────────────────────────────────────────┬────────────────────────┬────────────┬──────────────┬────────────────┬──────────────────┬─────────┬─────────┬───────────────────────────────────────┬────────────────────┬────────────────────┬─────────────────────────────────┬─────────────────────────────┐
│             resource_id              │ first_name  │ middle_name │ last_name  │ gender  │                   race                    │       ethnicity        │ birth_date │ phone_number │ marital_status │       city       │  state  │ country │            street_address             │      latitude      │     longitude      │ disablility_adjusted_life_years │ quality_adjusted_life_years │
│               varchar                │   varchar   │   varchar   │  varchar   │ varchar │                  varchar                  │        varchar         │  varchar   │   varchar    │    varchar     │     varchar 

In [35]:
con.sql(f"SELECT * FROM silver.synthea_encounter WHERE patient_id='{syn_id1}';")

┌──────────────────────────────────────┬──────────┬─────────┬────────────────┬────────────────────────────────┬──────────────────────────────────────┬───────────────────┬───────────────────┬─────────────────┬──────────────────────────────┬─────────────────────┬─────────────────────┬──────────────────────────────────────┬────────────────────────────────────┬──────────────────────────────────────┬────────────────────────────────────┐
│             resource_id              │  status  │  class  │ procedure_code │         procedure_name         │              patient_id              │ practitioner_code │ practitioner_type │ practitioner_id │      practitioner_name       │     start_date      │      end_date       │             location_id              │           location_name            │             provider_id              │           provider_name            │
│               varchar                │ varchar  │ varchar │    varchar     │            varchar             │               va

In [15]:
con.sql("SELECT DISTINCT marital_status FROM silver.synthea_patient;")

┌────────────────┐
│ marital_status │
│    varchar     │
├────────────────┤
│ Never Married  │
│ Widowed        │
│ Married        │
└────────────────┘

## Idea for Document

Patient Information
Name: {name}
Gender: {gender}
Birth Date: {birth_date}
Race: {race}
Ethnicity: {ethnicity}
Marital Status: {marital_status}

In [2]:
%watermark

Last updated: 2025-06-18T19:03:45.452311+08:00

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.31.0

Compiler    : MSC v.1938 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 20
Architecture: 64bit

